# DARF v2 — Pose-Aware, Block-Wise, Background-Locked

v1'in failure analysis'inden çıkardığımız 4 yeni mekanizma:

1. **Face-aware synthetic pose** — OpenPose iskeletine açık göz/kulak/burun-boyun lineları ekledim. Daenerys'in side-profile bias'ı pose ambiguity'sinden besleniyordu; bu bias'ı pose seviyesinde keser.
2. **Per-block mask floor** — Spatial LoRA gate floor'u tek scalar yerine `{down:0.15, mid:0.08, up:0.0}`. Erken/composition layer'larda yüksek floor (ortak ışık & arka plan), geç/identity layer'larda sıfır floor (saç/yüz leak yok).
3. **Background latent lock** — Üçüncü kişi hallüsinasyonunu önlemek için ilk %35 denoising step'inde pose mask dışındaki latent dondurulur.
4. **Evaluator entegrasyonu** — Diğer ekibin Hungarian-assignment ArcFace + MediaPipe Pose evaluator'ı kullanılır; v1 ile karşılaştırılabilir metric'ler.

**v1 notebook (`darf_experiments.ipynb`) sunum/inference için bozulmadan kalsın** — bu notebook v2 deneylerini içeriyor.

## Bölüm 1 — Setup

In [ ]:
import os
REPO_URL = "https://github.com/melik3kara/LoRA-scale-adaptive-feedback.git"
REPO_DIR = "/workspace/LoRA-scale-adaptive-feedback"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already at {REPO_DIR}, pulling latest...")
    !cd {REPO_DIR} && git pull

In [ ]:
# LoRA dosyaları (yeni kurulumda)
!pip install -q gdown
os.makedirs(f"{REPO_DIR}/data/loras", exist_ok=True)
DRIVE_FOLDER_ID = "1L6NA_AT5HGSNOcKjw7gECbPKnaN86Ix5"

import glob
if not glob.glob(f"{REPO_DIR}/data/loras/*.safetensors"):
    !gdown --folder "https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}" -O {REPO_DIR}/data/loras/
    import shutil
    for f in glob.glob(f"{REPO_DIR}/data/loras/**/*.safetensors", recursive=True):
        target = os.path.join(f"{REPO_DIR}/data/loras", os.path.basename(f))
        if f != target:
            shutil.move(f, target)
!ls -lh {REPO_DIR}/data/loras/

In [ ]:
# Dependencies — v2 ek olarak mediapipe ve scipy gerektirir
os.environ["HF_HOME"] = "/workspace/cache/huggingface"
os.environ["INSIGHTFACE_HOME"] = "/workspace/cache/insightface"
os.makedirs("/workspace/cache/huggingface", exist_ok=True)
os.makedirs("/workspace/cache/insightface", exist_ok=True)

!pip install -q diffusers==0.30.3 transformers==4.44.2 peft controlnet-aux insightface onnxruntime-gpu
!pip install -q "pydantic<2.10" "typing_extensions>=4.13.0"
!pip install -q mediapipe scipy   # v2 ek bağımlılıklar (Evaluator için)

In [ ]:
os.chdir(REPO_DIR)
import sys
if "src" not in sys.path:
    sys.path.insert(0, "src")
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

## Bölüm 2 — Pipeline + Scorers + Evaluator

Standart scorer'lara ek olarak diğer ekibin **Evaluator**'ı (Hungarian + MediaPipe) yükleniyor — sonuçlar onlarla karşılaştırılabilir.

In [ ]:
from pipeline import load_identities, build_pipeline
from scorer import FaceScorer
from pose_scorer import PoseScorer
from attribute_scorer import AttributeScorer
from evaluator import Evaluator
from PIL import Image

identities = load_identities()
pipe = build_pipeline(identities)

face_scorer = FaceScorer(reference_dir="data/reference_faces", device="cuda")
pose_scorer = PoseScorer()
attr_scorer = AttributeScorer(identities, device="cuda")
evaluator   = Evaluator(device="cuda")

# Reference embeddings for Evaluator (Hungarian assignment)
ref_embeds = {}
for k in identities:
    ref_path = identities[k]["reference_image"]
    ref_embeds[k] = evaluator.extract_face_embedding(Image.open(ref_path).convert("RGB"))
    print(f"Evaluator ref embedding loaded: {k}")

names = list(identities.keys())
identity_regions = {names[0]: (0, 0, 512, 1024), names[1]: (512, 0, 1024, 1024)}
print(f"\nReady. Identities={names}")

## Bölüm 3 — Face-Aware Pose Skeleton

v1'de kullanılan `two_person_pose.png` synthetic OpenPose iskeleti yüz yönünü tek nokta (nose) ile veriyordu — Daenerys LoRA bias'ı bunu side-profile olarak yorumluyordu. v2'de skeleton'a:
- **Eye-eye line** (yatay) → yüz düzleminin yönünü açıkça verir
- **Ear-ear line** → kafa genişliğini netler
- **Nose-neck line** → centreline
- Daha büyük face keypoint daireleri

ekledim. Bu ControlNet'e "yüz düzlemi öne bakıyor" sinyali verir.

Ablation için iki pose üretilir: **face-aware** ve **ambiguous** (face yapısı çökertilmiş).

In [ ]:
from face_aware_pose import make_face_aware_pose, get_face_aware_target_keypoints

pose_v1   = Image.open("data/pose_images/two_person_pose.png").convert("RGB")  # eski
pose_v2   = make_face_aware_pose(1024, 1024, front_facing=True)                # yeni
pose_amb  = make_face_aware_pose(1024, 1024, front_facing=False)               # ablation control

# Cache to disk for inspection
os.makedirs("data/pose_images", exist_ok=True)
pose_v2.save("data/pose_images/two_person_face_aware.png")
pose_amb.save("data/pose_images/two_person_ambiguous.png")

target_keypoints = get_face_aware_target_keypoints(front_facing=True)

from IPython.display import display
print("v1 pose (synthetic):");          display(pose_v1.resize((384, 384)))
print("v2 pose (face-aware):");         display(pose_v2.resize((384, 384)))
print("v2 pose (ambiguous control):");  display(pose_amb.resize((384, 384)))

## Bölüm 4 — Pose Ablation

**Hipotez:** Face-aware pose kullanırsak Daenerys'in side-profile bias'ı azalır.

Aynı seed, aynı LoRA scale, sadece pose değişiyor. Çıktıların yüz açılarını gözlemle.

In [ ]:
from pipeline import generate

common = dict(
    lora_scales={"hermione": 0.30, "daenerys": 1.10},
    ctrl_scale=0.95,
    seed=42,
    use_regional_attention=True,
    use_spatial_lora_gate=True,
    spatial_gate_block_floor={"down": 0.15, "mid": 0.08, "up": 0.0},
    identity_regions=identity_regions,
)

img_pose_v1  = generate(pipe, identities, pose_v1,  **common)
img_pose_v2  = generate(pipe, identities, pose_v2,  **common)
img_pose_amb = generate(pipe, identities, pose_amb, **common)

print("Pose v1 (eski, ambiguous nose):");    display(img_pose_v1.resize((512,512)))
print("Pose v2 (face-aware):");                display(img_pose_v2.resize((512,512)))
print("Pose ambiguous control (eyes collapsed):"); display(img_pose_amb.resize((512,512)))

## Bölüm 5 — Per-Block Floor Ablation

**Hipotez:** Floor erken/down layer'larda yüksek (composition coherence), geç/up layer'larda sıfır (identity isolation) tutarsa hem diptych hem leak'ten kaçınırız.

Floor configs (hepsi aynı seed=42, pose=v2):

In [ ]:
FLOOR_CONFIGS = {
    "hard":              {"down": 0.00, "mid": 0.00, "up": 0.00},   # v1 baseline
    "global_0.05":       {"down": 0.05, "mid": 0.05, "up": 0.05},   # v1 default
    "global_0.15":       {"down": 0.15, "mid": 0.15, "up": 0.15},   # diptych mitigation
    "per_block_v2":      {"down": 0.15, "mid": 0.08, "up": 0.00},   # v2 önerisi
    "per_block_strong":  {"down": 0.25, "mid": 0.10, "up": 0.00},   # daha agresif coherence
}

floor_results = {}
for label, bf in FLOOR_CONFIGS.items():
    print(f"\n=== floor config: {label} {bf} ===")
    img = generate(
        pipe, identities, pose_v2,
        lora_scales={"hermione": 0.30, "daenerys": 1.10},
        ctrl_scale=0.95,
        seed=42,
        use_regional_attention=True,
        use_spatial_lora_gate=True,
        spatial_gate_block_floor=bf,
        identity_regions=identity_regions,
    )
    floor_results[label] = img
    display(img.resize((448, 448)))

## Bölüm 6 — Background Latent Lock Ablation

**Hipotez:** Üçüncü-kişi hallüsinasyonu çoğunlukla pose iskeletinin DIŞINDA üretiliyor. İlk K denoising step'inde background latent'i dondurmak modelin oraya yeni figür inşa etmesini engeller.

Lock ratios:

In [ ]:
BG_CONFIGS = [0.00, 0.20, 0.35, 0.50]
bg_results = {}

for ratio in BG_CONFIGS:
    print(f"\n=== bg_lock_ratio={ratio} ===")
    img = generate(
        pipe, identities, pose_v2,
        lora_scales={"hermione": 0.30, "daenerys": 1.10},
        ctrl_scale=0.95,
        seed=42,
        use_regional_attention=True,
        use_spatial_lora_gate=True,
        spatial_gate_block_floor={"down": 0.15, "mid": 0.08, "up": 0.0},
        use_bg_lock=(ratio > 0),
        bg_lock_ratio=ratio,
        bg_lock_padding=24,
        identity_regions=identity_regions,
    )
    bg_results[f"lock_{ratio:.2f}"] = img
    display(img.resize((448, 448)))

**Extra-face metric:** her image'da kaç yüz var? Hedef: tam 2.

Üçüncü-kişi azaltma akademik olarak ölçülür.

In [ ]:
import numpy as np, cv2
def detected_face_count(img):
    img_bgr = cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2BGR)
    return len(evaluator.face_app.get(img_bgr))

for label, img in bg_results.items():
    n = detected_face_count(img)
    extra = max(0, n - 2)
    print(f"{label}: detected={n} faces  extra={extra}")

## Bölüm 7 — DARF v2 Full Pipeline

Hepsi birlikte: face-aware pose + per-block floor + bg-lock + dominance + spatial gate.

Multi-seed (3 seed) — istatistiksel olarak savunulabilir.

In [ ]:
from adaptive_loop import multi_lora_adaptive_generate

block_profile = {
    "hermione": {"down": 0.4, "mid": 0.7, "up": 1.0},
    "daenerys": {"down": 0.6, "mid": 1.1, "up": 1.3},
}

SEEDS = [42, 123, 777]
v2_results = {}

for seed in SEEDS:
    print(f"\n=========== DARF v2 — seed {seed} ===========")
    res = multi_lora_adaptive_generate(
        pipe, identities, pose_v2,
        face_scorer, pose_scorer, target_keypoints, identity_regions,
        ctrl_scale=0.95,
        id_threshold=0.35, pose_threshold=0.50,
        alpha_init={"hermione": 0.30, "daenerys": 1.10},
        alpha_min ={"hermione": 0.05, "daenerys": 0.50},
        alpha_max ={"hermione": 0.50, "daenerys": 1.70},
        delta_up=0.10, delta_down=0.05, delta_suppress=0.07,
        attribute_scorer=attr_scorer,
        block_profile=block_profile,
        max_iters=5,
        seed=seed,
        use_regional_attention=True,
        use_spatial_lora_gate=True,
    )
    v2_results[seed] = res
    display(res.image.resize((448, 448)))

## Bölüm 8 — Cross-Method Ablation Tablosu

Hem **kendi scorer'larımız** (FaceScorer dominance, AttributeScorer margin) hem de **diğer ekibin Evaluator'ı** (Hungarian-assigned ArcFace, MediaPipe pose error) ile aynı image'ları skorla — ekipler arası karşılaştırılabilirlik için.

In [ ]:
import pandas as pd

def collect_all_metrics(label, image, pose_ref):
    # Kendi scorer'larımız
    fc = face_scorer.score_image(image, identity_regions)
    pc = pose_scorer.score_image(image, target_keypoints, identity_regions)
    ac = attr_scorer.score_image(image, identity_regions)
    # Evaluator (Hungarian + MediaPipe)
    sim_h, sim_d = evaluator.detect_and_assign_faces(
        image, ref_embeds["hermione"], ref_embeds["daenerys"]
    )
    pose_err = evaluator.pose_keypoint_error(image, pose_ref)
    n_faces  = detected_face_count(image)

    return [{
        "method": label, "identity": "hermione",
        "arcface_self":  round(fc["hermione"]["arcface"],   4),
        "dominance":     round(fc["hermione"]["dominance"], 4),
        "pose":          round(pc["hermione"]["pose"],      4),
        "attr_margin":   round(ac["hermione"]["margin"],    4),
        "eval_arcface":  round(sim_h, 4),
        "eval_pose_err": round(pose_err, 4),
        "face_count":    n_faces,
    }, {
        "method": label, "identity": "daenerys",
        "arcface_self":  round(fc["daenerys"]["arcface"],   4),
        "dominance":     round(fc["daenerys"]["dominance"], 4),
        "pose":          round(pc["daenerys"]["pose"],      4),
        "attr_margin":   round(ac["daenerys"]["margin"],    4),
        "eval_arcface":  round(sim_d, 4),
        "eval_pose_err": round(pose_err, 4),
        "face_count":    n_faces,
    }]

rows = []
rows += collect_all_metrics("pose_v1",          img_pose_v1,  pose_v2)
rows += collect_all_metrics("pose_v2",          img_pose_v2,  pose_v2)
rows += collect_all_metrics("pose_ambiguous",   img_pose_amb, pose_v2)
for label, img in floor_results.items():
    rows += collect_all_metrics(f"floor_{label}", img, pose_v2)
for label, img in bg_results.items():
    rows += collect_all_metrics(f"bg_{label}", img, pose_v2)
for seed, res in v2_results.items():
    rows += collect_all_metrics(f"darf_v2_seed{seed}", res.image, pose_v2)

df = pd.DataFrame(rows)
os.makedirs("data/results/darf_v2", exist_ok=True)
df.to_csv("data/results/darf_v2/ablation.csv", index=False)
df

In [ ]:
# Aggregated mean ± std for DARF v2 (across seeds)
darf_rows = df[df["method"].str.startswith("darf_v2_seed")]
agg = darf_rows.groupby("identity").agg(["mean", "std"])
agg.to_csv("data/results/darf_v2/ablation_agg.csv")
print("DARF v2 multi-seed aggregate:")
agg

## Bölüm 9 — İmage Save + ZIP Download

In [ ]:
out_dir = "data/results/darf_v2"
os.makedirs(out_dir, exist_ok=True)

img_pose_v1.save(f"{out_dir}/pose_v1_synthetic.png")
img_pose_v2.save(f"{out_dir}/pose_v2_face_aware.png")
img_pose_amb.save(f"{out_dir}/pose_ambiguous.png")
for label, img in floor_results.items():
    img.save(f"{out_dir}/floor_{label}.png")
for label, img in bg_results.items():
    img.save(f"{out_dir}/bg_{label}.png")
for seed, res in v2_results.items():
    res.image.save(f"{out_dir}/darf_v2_seed{seed}.png")

# Pose images de
pose_v2.save(f"{out_dir}/skeleton_v2.png")
pose_amb.save(f"{out_dir}/skeleton_ambiguous.png")

import shutil
from IPython.display import FileLink
shutil.make_archive("/workspace/darf_v2_results", "zip", out_dir)
print("Zip ready: /workspace/darf_v2_results.zip")
FileLink("/workspace/darf_v2_results.zip")

## Notlar — Akademik Çerçeve

DARF v2 üç yeni mekanizma ile v1'in failure mode'larına doğrudan müdahale eder:

| Failure (v1) | v2 Mekanizması | Sebep |
|---|---|---|
| Daenerys side-profile | Face-aware pose | OpenPose nose-only signal'ı belirsiz; eye-eye + nose-neck lineları yön ekler |
| Diptych vs leak trade-off | Per-block floor | Compose-blocks'ta yüksek floor (coherence), face-blocks'ta sıfır (isolation) |
| Üçüncü kişi hallüsinasyonu | BG latent lock | İlk %35 step'te background latent dondurulur, model orada figür yaratmaz |
| Cross-team karşılaştırılabilirlik | Evaluator entegrasyonu | Hungarian ArcFace + MediaPipe pose, diğer ekiple aynı metric |

Bu mekanizmalar ortogonal — ablation tablosu (Bölüm 8) her birinin tek başına ve birlikte etkisini ölçer.